# Stock Forecasting with Machine Learning
## Learning Candlestick and Volume Patterns for S&P 500 Prediction

**Final Production Market Forecasting System**  
**Tien Le**  
Copyright © 2026 by Tien Le


> **Public Demo Version**
>
> This notebook preserves the original data workflow, model names, training, evaluation, results, and conclusions.
> Only proprietary feature-building formulas and original feature names are omitted where applicable.


# Table of Contents

1. [Introduction](#Introduction)
2. [Motivation](#Motivation)
3. [Research Questions](#Research-Questions)
4. [Project Evolution](#Project-Evolution)
5. [Dataset Description](#Dataset-Description)
6. [Feature Engineering](#Feature-Engineering)
7. [Avoiding Data Leakage](#Avoiding-Data-Leakage)
8. [Models Compared](#Models-Compared)
9. [Evaluation Metrics](#Evaluation-Metrics)
10. [Final Results](#Final-Results)
11. [Today's Prediction](#Todays-Prediction)
12. [Prediction Tracking](#Prediction-Tracking)
13. [Limitations](#Limitations)
14. [Future Work](#Future-Work)
15. [Final Conclusion](#Final-Conclusion)


# Introduction

Over the past several years I have actively traded the stock market using a combination of fundamental analysis, macroeconomic events, and investor psychology. Rather than relying primarily on technical indicators, my investment decisions have focused on understanding businesses, evaluating long-term value, and recognizing situations where market sentiment becomes excessively optimistic or pessimistic.

My trading journey evolved through several stages.

During my first year, I mainly traded highly volatile stocks such as Tesla and Coinbase using a simple "buy low, sell high" strategy. My portfolio briefly tripled within a few months, but much of that success was driven by favorable market conditions rather than a repeatable process. Shortly afterward, I experienced a large drawdown of nearly 30% in just a few days after taking highly leveraged short-term options positions, losing several hundred thousand dollars. Although I still finished the year with approximately 2.5× portfolio growth, the experience taught me that high returns without consistency are not sustainable.

Over the following two years, I adopted a more conservative approach by selling covered calls and cash-secured puts instead of aggressively buying options. This strategy produced more consistent results while reducing overall portfolio volatility. My investment decisions continued to rely heavily on fundamentals, market sentiment, and macroeconomic events such as tariff announcements, geopolitical conflicts, volatility spikes, and company-specific news. My general philosophy became:
- Buy when fear is high.
- Sell when excitement is high.
- Focus on business quality rather than short-term news.
- Size positions according to conviction rather than emotion.

One area where I had relatively little experience was technical analysis. Instead of performing chart analysis myself, I often subscribed to professional technical analysis services and AI trading assistants. I found that although technical analysis rarely determined whether I entered a trade, it frequently influenced position sizing. When both fundamentals and technical signals agreed, I was willing to allocate significantly more capital because the probability of success appeared higher.

This naturally led to the main question behind this project.

If experienced traders can identify useful information from candlestick charts, then machine learning—which excels at pattern recognition—may be able to discover these patterns more consistently and objectively than humans.

Rather than relying entirely on external AI trading services, I wanted to build my own forecasting model. Even if my model is not the most accurate available, understanding exactly how it reaches its predictions provides confidence and transparency when making investment decisions.


# Motivation

This project focuses on the S&P 500 through the SPY exchange-traded fund because it is highly liquid, widely followed, and broadly reflects the U.S. equity market. It also provides a more diversified forecasting target than an individual high-volatility stock.

A passive investment in the S&P 500 has historically been difficult to beat. In the data used for this project, approximately 55% of trading days are positive and 45% are negative. Therefore, a model that simply predicts **Up** every day can already achieve approximately 55% accuracy. Any directional forecasting model must be compared with this baseline rather than judged only by its raw accuracy.

However, overall accuracy is not the only possible source of value. A model may still be useful if it identifies a meaningful portion of the largest down days. Such warnings could help reduce exposure to SPY or leveraged products such as SPXL before unusually large declines. Even a modest forecasting advantage could improve risk management and contribute to portfolio diversification when combined with careful position sizing.

This notebook is an exploratory machine-learning study, not a claim of guaranteed investment performance. The objective is to test whether historical candlestick, price-action, volume, and technical features contain measurable information about the following trading day.


# Research Questions

1. Can traditional time-series models predict the exact next-day SPY closing price better than a simple persistence forecast?
2. Can machine-learning classifiers outperform the **Always Up** baseline when predicting next-day direction?
3. Do broader market variables such as VIX, QQQ, Treasury yields, and Bitcoin add useful predictive information?
4. Can candlestick and price-structure features help identify unusually large down days?
5. Do more complicated feature sets improve generalization, or do they introduce additional noise and overfitting?
6. Can an interpretable model provide practical warning signals that support position sizing and portfolio risk management?


# Project Evolution

## Experimental Version 1 — Exact Next-Day Closing Price

I first used AutoReg, ARIMA, and SARIMAX to forecast the next SPY closing price. None produced meaningful improvement over a simple persistence forecast that assumed tomorrow's price would be close to today's price. This suggested that exact next-day price prediction was too difficult for the traditional time-series methods tested here.

## Experimental Version 2 — Next-Day Up/Down Classification

I reformulated the problem as classification and compared Logistic Regression, Lasso, Ridge, Elastic Net, Decision Tree, Random Forest, and XGBoost. Logistic Regression produced only a small improvement in some experiments. Lasso removed all features, while Ridge, Elastic Net, and Random Forest often predicted **Up** every day. XGBoost frequently overfit the training data. A small Decision Tree was the strongest and most interpretable model.

## Experimental Version 3 — Broader Market Information

I added features from VIX, QQQ, the 10-Year Treasury yield, and Bitcoin. These variables did not consistently improve performance and sometimes made it worse. One possible explanation is that much of this public information is already reflected in SPY's current price, while the additional variables introduce noise.

## Experimental Version 4 — Simplified Candlestick Features and Large Moves

I returned to SPY-only information and engineered simpler features that more closely resemble how a trader reads a chart. The target was expanded beyond ordinary Up/Down prediction to include unusually large market declines. This produced the most meaningful results: the strongest models identified approximately 75%–80% of the target large-down days in the historical test period.

## Experimental Version 5 — Extended and More Complicated Features

I created a larger and more detailed technical feature set. The additional complexity did not improve the results. In particular, recall and precision for larger down days were generally no better than with the simpler feature set. This suggested that the extended variables added noise and increased the risk of overfitting.

## Final Production System — Final Model and Prediction System

The final version returns to compressed candlestick-structure features and emphasizes interpretability, chronological testing, leakage prevention, probability estimates, model comparison, daily predictions, and prediction tracking. The principal objective is not merely to maximize overall accuracy, but to generate useful warnings for unusually large down days.


# Dataset Description

## SPY

SPY, the SPDR S&P 500 ETF Trust, is used as the forecasting target and as a practical proxy for the S&P 500. The model uses daily open, high, low, close, and volume data.

## Data Source: Yahoo Finance

Historical SPY market data are downloaded with the `yfinance` Python package. The notebook cleans the returned columns before generating features.

## Date Range

The dataset begins on **January 1, 2017**. This period includes several distinct market environments, including the pre-COVID market, the 2020 crash and recovery, the 2022 bear market, the subsequent technology-driven bull market, tariff-related volatility, and recent market conditions.


## Train/Test Split

Because this is time-series data, observations are kept in chronological order. Earlier observations form the training set, while approximately the most recent year forms the out-of-sample test set. Random shuffling is not used because it would allow future market regimes to influence evaluation of earlier observations.


# Feature Engineering

The final feature set summarizes recent candlestick and market structure rather than supplying every raw candle independently. It includes information such as:

- bullish and bearish candle ratios;
- average body, upper-shadow, lower-shadow, and range sizes;
- consecutive up-day and down-day streaks;
- higher-high, higher-low, lower-high, and lower-low behavior;
- short-, medium-, and longer-window price structure;
- weekly and monthly market context derived from completed higher-timeframe candles;
- volume and volatility behavior where applicable.

This compressed design aims to represent chart structure while controlling dimensionality and reducing noise.


# Avoiding Data Leakage

Data leakage is especially dangerous in financial forecasting because a small amount of future information can create results that appear profitable but cannot be reproduced in live trading.

The notebook uses the following safeguards:

- Features for date *t* are constructed only from information available on or before date *t*.
- The target is based on the return from date *t* to date *t + 1*.
- Target thresholds for large moves are calculated from the training period rather than from the complete dataset.
- Train and test observations remain in chronological order.
- Scaling or model fitting is performed on the training set before being applied to the test set.
- The final row is treated separately when generating a live prediction because its next-day outcome is not yet known.

These rules help ensure that test results approximate how the models would have operated using only information available at the time.


# Models Compared

The experiments compare models with different levels of complexity and interpretability:

- Logistic Regression
- Lasso Logistic Regression
- Ridge Logistic Regression
- Elastic Net Logistic Regression
- Decision Tree
- Random Forest
- XGBoost
- AutoReg, ARIMA, and SARIMAX in earlier price-forecasting versions

Decision Trees receive particular attention in the final notebook because they performed well relative to the other classifiers, remained interpretable, and allowed the prediction rules to be displayed directly.


# Evaluation Metrics

No single metric is sufficient for an imbalanced market-warning problem.

- **Accuracy:** the proportion of all trading days classified correctly. This is compared with the Always Up baseline.
- **Precision:** among the days predicted as a target down day, the proportion that actually became target down days.
- **Recall:** among all actual target down days, the proportion successfully identified by the model.
- **F1 Score:** the harmonic mean of precision and recall.
- **Confusion Matrix:** the counts of true positives, true negatives, false positives, and false negatives.

For large-down warnings, recall is particularly important because a missed decline may be costly. Precision also matters because too many false warnings can cause a strategy to exit the market unnecessarily. The best model therefore depends on the intended trading and risk-management objective rather than on accuracy alone.


In [1]:
# Packages
%pip install xgboost
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.tree import DecisionTreeClassifier, plot_tree, DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report)
import warnings
warnings.filterwarnings("ignore")
from IPython.display import Markdown, display

Note: you may need to restart the kernel to use updated packages.


# Data Collection and Preparation

In [2]:
# No public data-download package is required.
# The private preparation notebook creates the reproducible demo snapshot.


In [3]:
# Live download moved to PRIVATE-Prepare-Demo-Data.ipynb.


## Choosing the Historical Data Window for Prediction
When predicting tomorrow's SPY closing price, using more historical data is not always better. In many cases, 20 years of data does not provide significantly better predictions than 5–10 years of data. In fact, older data can sometimes reduce performance because market conditions, trading behavior, regulations, and economic environments change over time.

For this reason, many quantitative trading strategies are developed using approximately 3–10 years of historical data rather than the entire available history.

### Why I Chose a Start Date of 2017-01-01
I selected a start date of 2017-01-01, which provides approximately 9.5 years of market data. This period captures several important market regimes and major economic events, including:

- Trump's first presidential term (2017–2021)
- The COVID-19 market crash and recovery
- The Biden administration years
- The 2022 bear market
- The AI-driven bull market beginning in 2023
- Trump's second presidential term beginning in 2025

This time period is long enough to include multiple market environments while remaining recent enough to be relevant for forecasting current market behavior.

## Implementing the Compressed Feature Set

In this version, we changed from using many raw lagged candle values to using compressed candlestick structure features.

Instead of giving the model every candle from the last 60 days individually, we summarized recent price behavior using features such as:

- Bullish and bearish candle ratios over recent windows
- Average candle body size
- Average upper and lower shadow size
- Average candle range
- Up-day and down-day streaks
- Higher-high, higher-low, lower-high, and lower-low streaks
- Distance from recent highs and lows
- Volume ratio and volume changes
- ATR and Bollinger Band volatility features
- Weekly and monthly candlestick context

The goal was to represent the chart more like how a trader reads it: not as isolated candles, but as trend, momentum, volatility, volume, and recent market structure.

In [4]:
# Public-demo feature loader
# The private feature formulas are intentionally not included.
from pathlib import Path

def locate_demo_file(filename):
    for path in [Path(filename), Path("data") / filename]:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"{filename} was not found. Run PRIVATE-Prepare-Demo-Data.ipynb, "
        "then place the generated CSV beside this notebook or in data/."
    )

def load_private_feature_snapshot(filename):
    snapshot = pd.read_csv(
        locate_demo_file(filename),
        parse_dates=["Date"]
    ).set_index("Date").sort_index()
    required = {"Target", "Next_Day_Return", "Market_Close"}
    missing = required.difference(snapshot.columns)
    if missing:
        raise ValueError(f"Missing snapshot columns: {sorted(missing)}")
    feature_cols = [c for c in snapshot if c.startswith("Feature_")]
    if not feature_cols:
        raise ValueError("No anonymized Feature_ columns were found.")
    return snapshot, feature_cols


In [5]:
snapshot, feature_cols = load_private_feature_snapshot("demo_features_experiment4_final.csv")
v_structure_prediction = snapshot[feature_cols + ["Target"]].copy()
v_structure = v_structure_prediction.dropna(subset=["Target"]).copy()
future_return_snapshot = snapshot["Next_Day_Return"].copy()
market_close = snapshot["Market_Close"].copy()
spy_clean = pd.DataFrame({"Close": market_close}).dropna()
print("Private feature definitions hidden; shared Experiment 4 / Final snapshot loaded.")
print("Number of features:", len(feature_cols))


Private feature definitions hidden; shared Experiment 4 / Final snapshot loaded.
Number of features: 253


# Model Experiments

## Experiment 1: Predict Next-Day Direction

### Now create the data set

In [6]:
# This is the same feature matrix used by Experiment 4.
print(v_structure.shape)
print(v_structure["Target"].value_counts())


(1952, 254)
Target
1.0    1074
0.0     878
Name: count, dtype: int64


In [7]:
X = v_structure.drop(columns=["Target"])
y = v_structure["Target"].astype(int)

test_start = X.index.max() - pd.DateOffset(years=1)

X_train = X[X.index < test_start]
X_test = X[X.index >= test_start]

y_train = y[y.index < test_start]
y_test = y[y.index >= test_start]

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts())

print("\nTest target distribution:")
print(y_test.value_counts())

always_up_accuracy = (y_test == 1).mean()

print(f"\nAlways Up Accuracy: {always_up_accuracy:.4f}")
print(f"Test period: {X_test.index.min().date()} to {X_test.index.max().date()}")

Train shape: (1700, 253)
Test shape : (252, 253)

Train target distribution:
Target
1    935
0    765
Name: count, dtype: int64

Test target distribution:
Target
1    139
0    113
Name: count, dtype: int64

Always Up Accuracy: 0.5516
Test period: 2025-07-09 to 2026-07-09


## Train Decision Tree

In [8]:
tree_structure = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=20,
    min_samples_split=5,
    random_state=42
)

tree_structure.fit(X_train, y_train)

train_pred = tree_structure.predict(X_train)
test_pred = tree_structure.predict(X_test)

train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)
always_up_acc = (y_test == 1).mean()

cm = confusion_matrix(y_test, test_pred)

predicted_down = (test_pred == 0).sum()
correct_down = cm[0, 0]

print("========== Decision Tree: Compressed Candle Structure Features ==========")
print("Train Accuracy:", train_acc)
print("Test Accuracy :", test_acc)
print("Always Up Accuracy:", always_up_acc)

print("\nClassification Report:")
print(classification_report(y_test, test_pred))

print("\nConfusion Matrix:")
print(cm)

print("\nPrediction Summary:")
print("Predicted Down:", predicted_down)
print("Correctly predicted Down days:", correct_down)

print("\nObservation:")
print(
    f"The decision tree predicted {predicted_down} down days, "
    f"and {correct_down} of them were correct. "
    f"The test accuracy changed from {always_up_acc:.4f} "
    f"to {test_acc:.4f}."
)

========== Decision Tree: Compressed Candle Structure Features ==========
Train Accuracy: 0.5858823529411765
Test Accuracy : 0.5714285714285714
Always Up Accuracy: 0.5515873015873016

Classification Report:
              precision    recall  f1-score   support

           0       0.63      0.11      0.18       113
           1       0.57      0.95      0.71       139

    accuracy                           0.57       252
   macro avg       0.60      0.53      0.45       252
weighted avg       0.60      0.57      0.47       252


Confusion Matrix:
[[ 12 101]
 [  7 132]]

Prediction Summary:
Predicted Down: 19
Correctly predicted Down days: 12

Observation:
The decision tree predicted 19 down days, and 12 of them were correct. The test accuracy changed from 0.5516 to 0.5714.


In [9]:
display(Markdown(
    f"""
### Result Summary

The compressed candle-structure Decision Tree performed better than the Always Up baseline.

The test accuracy improved from **{always_up_acc:.4f}** to **{test_acc:.4f}**.

The model predicted **{predicted_down}** down days, and **{correct_down}** of them were correct.
"""
))


### Result Summary

The compressed candle-structure Decision Tree performed better than the Always Up baseline.

The test accuracy improved from **0.5516** to **0.5714**.

The model predicted **19** down days, and **12** of them were correct.


## Experiment 2: Predict Large Down Moves

### Try different models to see which predict big Down days best

In [10]:
# Optional XGBoost
try:
    from xgboost import XGBClassifier
    has_xgb = True
except ImportError:
    has_xgb = False


# =====================================================
# 1. Prepare features and next-day returns
# =====================================================

future_return = snapshot["Next_Day_Return"].copy()

# Use v_structure features, but remove its previous Target column
base_X = v_structure.drop(
    columns=["Target"],
    errors="ignore"
).copy()

# Align features and future returns
common_index = base_X.index.intersection(future_return.dropna().index)

base_X = base_X.loc[common_index].copy()
future_return = future_return.loc[common_index].copy()

# Remove any remaining invalid values
base_X = base_X.replace([np.inf, -np.inf], np.nan)

valid_rows = base_X.notna().all(axis=1) & future_return.notna()

base_X = base_X.loc[valid_rows]
future_return = future_return.loc[valid_rows]

print("Available observations:", len(base_X))
print("Number of features:", base_X.shape[1])


# =====================================================
# 2. Fixed train/test split
# Most recent calendar year is the test period
# =====================================================

test_start = base_X.index.max() - pd.DateOffset(years=1)

X_train = base_X[base_X.index < test_start]
X_test = base_X[base_X.index >= test_start]

train_future_return = future_return.loc[X_train.index]
test_future_return = future_return.loc[X_test.index]

print("\nTrain shape:", X_train.shape)
print("Test shape :", X_test.shape)

print(
    "Test period:",
    X_test.index.min().date(),
    "to",
    X_test.index.max().date()
)


# =====================================================
# 3. Models to compare
# =====================================================

models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=3,
        min_samples_leaf=20,
        min_samples_split=5,
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        max_depth=5,
        min_samples_leaf=20,
        min_samples_split=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "Logistic Regression": LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        random_state=42
    )
}

if has_xgb:
    models["Simple XGBoost"] = XGBClassifier(
        objective="binary:logistic",
        n_estimators=100,
        learning_rate=0.03,
        max_depth=2,
        subsample=0.7,
        colsample_bytree=0.7,
        min_child_weight=10,
        gamma=1,
        reg_alpha=1,
        reg_lambda=10,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    )


# =====================================================
# 4. Target definitions
# =====================================================

target_settings = {
    "Top 5% Down": 0.05,
    "Top 10% Down": 0.10,
    "Top 15% Down": 0.15,
    "Top 20% Down": 0.20,
    "Top 25% Down": 0.25,
    "Top 30% Down": 0.30,
    "Top 35% Down": 0.35,
    "Top 40% Down": 0.40,
    "Top 45% Down": 0.45,
    "All Down Days": "all_down"
}


# =====================================================
# 5. Run all experiments
# =====================================================

results = []

for target_name, q in target_settings.items():

    # -------------------------------------------------
    # Create target
    # Threshold is calculated from TRAINING returns only
    # -------------------------------------------------
    if q == "all_down":
        threshold = 0.0

        y_train = (train_future_return < 0).astype(int)
        y_test = (test_future_return < 0).astype(int)

    else:
        threshold = train_future_return.quantile(q)

        y_train = (
            train_future_return <= threshold
        ).astype(int)

        y_test = (
            test_future_return <= threshold
        ).astype(int)

    for model_name, model_template in models.items():

        # Create a fresh copy of each model
        model = clone(model_template)

        # Train
        model.fit(X_train, y_train)

        # Predict and preserve the date index
        test_pred = pd.Series(
            model.predict(X_test),
            index=X_test.index,
            name="Prediction"
        ).astype(int)

        # -------------------------------------------------
        # Confusion matrix
        # labels=[0, 1] guarantees a 2x2 matrix
        # -------------------------------------------------
        cm = confusion_matrix(
            y_test,
            test_pred,
            labels=[0, 1]
        )

        tn, fp, fn, tp = cm.ravel()

        # -------------------------------------------------
        # Standard classification metrics
        # -------------------------------------------------
        accuracy = accuracy_score(y_test, test_pred)

        precision = precision_score(
            y_test,
            test_pred,
            zero_division=0
        )

        recall = recall_score(
            y_test,
            test_pred,
            zero_division=0
        )

        f1 = f1_score(
            y_test,
            test_pred,
            zero_division=0
        )

        always_no_accuracy = (y_test == 0).mean()

        # -------------------------------------------------
        # Target-event counts
        # -------------------------------------------------
        actual_down = int(y_test.sum())
        predicted_down = int(test_pred.sum())
        correct_down = int(tp)

        # -------------------------------------------------
        # Practical direction check
        #
        # Among every day predicted as Top-X% Down:
        # - How many actually fell at all?
        # - How many actually rose?
        # -------------------------------------------------
        predicted_down_mask = test_pred == 1

        predicted_any_down = int(
            (
                predicted_down_mask
                & (test_future_return < 0)
            ).sum()
        )

        predicted_up = int(
            (
                predicted_down_mask
                & (test_future_return >= 0)
            ).sum()
        )

        if predicted_down > 0:
            down_hit_rate = (
                predicted_any_down / predicted_down
            )
        else:
            down_hit_rate = np.nan

        # -------------------------------------------------
        # Store results using consistent column names
        # -------------------------------------------------
        results.append({
            "Target": target_name,
            "Threshold": float(threshold),
            "Model": model_name,

            "Actual Down Days": actual_down,
            "Predicted Down Days": predicted_down,
            "Correct Down Days": correct_down,

            "Predicted Days That Were Down": predicted_any_down,
            "Predicted Days That Were Up": predicted_up,
            "Down Hit Rate": down_hit_rate,

            "Precision": precision,
            "Recall": recall,
            "F1": f1,

            "Accuracy": accuracy,
            "Always-No Accuracy": always_no_accuracy,

            "True Negatives": int(tn),
            "False Positives": int(fp),
            "False Negatives": int(fn),
            "True Positives": int(tp)
        })


# =====================================================
# 6. Results table
# =====================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by=["Recall", "Precision", "F1"],
    ascending=False
).reset_index(drop=True)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

display(results_df)





Available observations: 1952
Number of features: 253

Train shape: (1700, 253)
Test shape : (252, 253)
Test period: 2025-07-09 to 2026-07-09


,Target,Threshold,Model,Actual Down Days,Predicted Down Days,Correct Down Days,Predicted Days That Were Down,Predicted Days That Were Up,Down Hit Rate,Precision,Recall,F1,Accuracy,Always-No Accuracy,True Negatives,False Positives,False Negatives,True Positives
0,Top 20% Down,-0.006707,Decision Tree,33,143,26,67,76,0.468531,0.181818,0.787879,0.295455,0.507937,0.869048,102,117,7,26
1,Top 25% Down,-0.004979,Decision Tree,44,140,34,68,72,0.485714,0.242857,0.772727,0.369565,0.539683,0.825397,102,106,10,34
2,Top 40% Down,-0.001039,Decision Tree,94,153,63,72,81,0.470588,0.411765,0.670213,0.510121,0.519841,0.626984,68,90,31,63
3,Top 30% Down,-0.003223,Decision Tree,63,135,39,62,73,0.459259,0.288889,0.619048,0.393939,0.523810,0.750000,93,96,24,39
4,Top 15% Down,-0.008823,Decision Tree,25,101,15,47,54,0.465347,0.148515,0.600000,0.238095,0.619048,0.900794,141,86,10,15
5,All Down Days,0.000000,Decision Tree,113,145,67,67,78,0.462069,0.462069,0.592920,0.519380,0.507937,0.551587,61,78,46,67
6,Top 45% Down,0.000016,Decision Tree,113,143,63,63,80,0.440559,0.440559,0.557522,0.492188,0.484127,0.551587,59,80,50,63
7,Top 10% Down,-0.012938,Decision Tree,15,67,7,26,41,0.388060,0.104478,0.466667,0.170732,0.730159,0.940476,177,60,8,7
8,Top 20% Down,-0.006707,Random Forest,33,50,13,22,28,0.440000,0.260000,0.393939,0.313253,0.773810,0.869048,182,37,20,13
9,Top 25% Down,-0.004979,Random Forest,44,62,16,29,33,0.467742,0.258065,0.363636,0.301887,0.706349,0.825397,162,46,28,16


In [11]:
# Automatically find the top three Recall results
# -------------------------------------------------------

top3 = results_df.head(3).reset_index(drop=True)

r1 = top3.iloc[0]
r2 = top3.iloc[1]
r3 = top3.iloc[2]

# Final Results

The following output ranks the experiments primarily by their ability to detect target down days. The notebook reports recall, precision, F1 score, the number of actual target days caught, and the number of warnings generated. These results should be interpreted relative to the target definition, test period, and Always Up baseline.


In [12]:
display(Markdown(
    f"""
# Conclusion

Among all experiments, the models with the **highest ability to detect target down days, as measured by recall,** were:

### 🥇 Best Model
- **Target:** {r1['Target']}
- **Model:** {r1['Model']}
- **Recall:** **{r1['Recall']:.1%}**
- **Precision:** **{r1['Precision']:.1%}**
- **F1 score:** **{r1['F1']:.3f}**
- Correctly detected **{int(r1['Correct Down Days'])} of {int(r1['Actual Down Days'])}** target down days.
- Predicted **{int(r1['Predicted Down Days'])}** target down days in total.
- Of those predictions, **{int(r1['Predicted Days That Were Down'])}** were followed by an actual down day.
- **Down Hit Rate:** **{r1['Down Hit Rate']:.1%}**

### 🥈 Second-Best Model
- **Target:** {r2['Target']}
- **Model:** {r2['Model']}
- **Recall:** **{r2['Recall']:.1%}**
- **Precision:** **{r2['Precision']:.1%}**
- **F1 score:** **{r2['F1']:.3f}**
- Correctly detected **{int(r2['Correct Down Days'])} of {int(r2['Actual Down Days'])}** target down days.
- Predicted **{int(r2['Predicted Down Days'])}** target down days in total.
- Of those predictions, **{int(r2['Predicted Days That Were Down'])}** were followed by an actual down day.
- **Down Hit Rate:** **{r2['Down Hit Rate']:.1%}**

### 🥉 Third-Best Model
- **Target:** {r3['Target']}
- **Model:** {r3['Model']}
- **Recall:** **{r3['Recall']:.1%}**
- **Precision:** **{r3['Precision']:.1%}**
- **F1 score:** **{r3['F1']:.3f}**
- Correctly detected **{int(r3['Correct Down Days'])} of {int(r3['Actual Down Days'])}** target down days.
- Predicted **{int(r3['Predicted Down Days'])}** target down days in total.
- Of those predictions, **{int(r3['Predicted Days That Were Down'])}** were followed by an actual down day.
- **Down Hit Rate:** **{r3['Down Hit Rate']:.1%}**

## Overall Findings

- The best-performing model was **{r1['Model']}**, trained to predict **{r1['Target']}**.

- It achieved **{r1['Recall']:.1%} recall**, meaning that it detected **{int(r1['Correct Down Days'])} of the {int(r1['Actual Down Days'])}** actual target down days.

- It achieved **{r1['Precision']:.1%} precision**, meaning that **{r1['Precision']:.1%}** of its target-down predictions were severe enough to qualify as **{r1['Target']}**.

- The model issued **{int(r1['Predicted Down Days'])}** warnings. Of these:
  - **{int(r1['Correct Down Days'])}** correctly qualified as **{r1['Target']}**.
  - **{int(r1['Predicted Days That Were Down'])}** resulted in some type of down day, including less-severe declines.
  - **{int(r1['Predicted Days That Were Up'])}** resulted in an up or flat day.
  - The resulting **Down Hit Rate was {r1['Down Hit Rate']:.1%}**.

- These results suggest that compressed candlestick-structure features may contain useful information for identifying periods of market weakness.

- However, the model still produces false alarms and should be interpreted as an **early-warning or risk-management indicator**, not as a guaranteed trading signal.

- Such a warning may be useful when evaluating whether to reduce exposure to a daily leveraged bullish ETF such as **SPXL**, where market declines are magnified.

- A simple way to do this
"""))


# Conclusion

Among all experiments, the models with the **highest ability to detect target down days, as measured by recall,** were:

### 🥇 Best Model
- **Target:** Top 20% Down
- **Model:** Decision Tree
- **Recall:** **78.8%**
- **Precision:** **18.2%**
- **F1 score:** **0.295**
- Correctly detected **26 of 33** target down days.
- Predicted **143** target down days in total.
- Of those predictions, **67** were followed by an actual down day.
- **Down Hit Rate:** **46.9%**

### 🥈 Second-Best Model
- **Target:** Top 25% Down
- **Model:** Decision Tree
- **Recall:** **77.3%**
- **Precision:** **24.3%**
- **F1 score:** **0.370**
- Correctly detected **34 of 44** target down days.
- Predicted **140** target down days in total.
- Of those predictions, **68** were followed by an actual down day.
- **Down Hit Rate:** **48.6%**

### 🥉 Third-Best Model
- **Target:** Top 40% Down
- **Model:** Decision Tree
- **Recall:** **67.0%**
- **Precision:** **41.2%**
- **F1 score:** **0.510**
- Correctly detected **63 of 94** target down days.
- Predicted **153** target down days in total.
- Of those predictions, **72** were followed by an actual down day.
- **Down Hit Rate:** **47.1%**

## Overall Findings

- The best-performing model was **Decision Tree**, trained to predict **Top 20% Down**.

- It achieved **78.8% recall**, meaning that it detected **26 of the 33** actual target down days.

- It achieved **18.2% precision**, meaning that **18.2%** of its target-down predictions were severe enough to qualify as **Top 20% Down**.

- The model issued **143** warnings. Of these:
  - **26** correctly qualified as **Top 20% Down**.
  - **67** resulted in some type of down day, including less-severe declines.
  - **76** resulted in an up or flat day.
  - The resulting **Down Hit Rate was 46.9%**.

- These results suggest that compressed candlestick-structure features may contain useful information for identifying periods of market weakness.

- However, the model still produces false alarms and should be interpreted as an **early-warning or risk-management indicator**, not as a guaranteed trading signal.

- Such a warning may be useful when evaluating whether to reduce exposure to a daily leveraged bullish ETF such as **SPXL**, where market declines are magnified.

- A simple way to do this


# Today's Prediction

In [13]:
# Use the existing v_structure dataframe
# =====================================================

data = v_structure.copy()

# Separate existing features from the target
X_all = data.drop(
    columns=["Target"],
    errors="ignore"
).copy()

X_all = X_all.replace(
    [np.inf, -np.inf],
    np.nan
)

# Remove rows with incomplete features
X_all = X_all.dropna()

print("Latest SPY date       :", market_close.dropna().index.max().date())
print("Latest feature date   :", X_all.index.max().date())
print("Number of features    :", X_all.shape[1])

Latest SPY date       : 2026-07-10
Latest feature date   : 2026-07-09
Number of features    : 253


## Update the function to not removing today date

In [14]:
# The prediction-inclusive feature matrix was already exported by the private notebook.
# Its final row has Target = NaN because the next trading-session result is unknown.


In [15]:
# Reuse the prediction-inclusive form of the same canonical snapshot.
v_structure = v_structure_prediction.copy()
print("Latest SPY date       :", market_close.dropna().index.max().date())
print("Latest feature date   :", v_structure.index.max().date())
print("Number of features    :", len(feature_cols))
print(v_structure[["Target"]].tail())


Latest SPY date       : 2026-07-10
Latest feature date   : 2026-07-10
Number of features    : 253
            Target
Date              
2026-07-06     0.0
2026-07-07     0.0
2026-07-08     1.0
2026-07-09     1.0
2026-07-10     NaN


In [16]:
# Separate training rows from prediction row
# =====================================================

# Historical rows with known next-day results
train_data = v_structure[
    v_structure["Target"].notna()
].copy()

X_train = train_data.drop(
    columns=["Target"]
)

y_train = train_data[
    "Target"
].astype(int)


# Latest row with unknown next-day result
prediction_data = v_structure[
    v_structure["Target"].isna()
].copy()

if prediction_data.empty:
    raise ValueError(
        "No row with an unknown future target was found."
    )

# Use only the latest unknown row
X_tomorrow = (
    prediction_data
    .drop(columns=["Target"])
    .tail(1)
)


print("Training shape       :", X_train.shape)
print("Prediction shape     :", X_tomorrow.shape)

print(
    "Training period      :",
    X_train.index.min().date(),
    "to",
    X_train.index.max().date()
)

print(
    "Prediction uses date :",
    X_tomorrow.index[0].date()
)

print("\nTraining target distribution:")
print(y_train.value_counts())

print(
    "\nHistorical Up rate:",
    f"{y_train.mean():.2%}"
)

Training shape       : (1952, 253)
Prediction shape     : (1, 253)
Training period      : 2018-10-01 to 2026-07-09
Prediction uses date : 2026-07-10

Training target distribution:
Target
1    1074
0     878
Name: count, dtype: int64

Historical Up rate: 55.02%


## First: Predict Up/Down for SP500 tomorrow by Decision Tree

In [17]:
# Train Decision Tree on all known history
# =====================================================

final_tree = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=20,
    min_samples_split=5,
    class_weight="balanced",
    random_state=42
)

final_tree.fit(
    X_train,
    y_train
)

print("Decision Tree trained successfully.")

Decision Tree trained successfully.


In [18]:
# Predict tomorrow Up or Down
# =====================================================

prediction = int(
    final_tree.predict(X_tomorrow)[0]
)

predicted_probabilities = final_tree.predict_proba(
    X_tomorrow
)[0]

probability_by_class = dict(
    zip(
        final_tree.classes_,
        predicted_probabilities
    )
)

probability_down = probability_by_class.get(0, 0.0)
probability_up = probability_by_class.get(1, 0.0)

prediction_label = (
    "UP"
    if prediction == 1
    else "DOWN"
)

feature_date = X_tomorrow.index[0]

latest_close = float(
    spy_clean.loc[
        feature_date,
        "Close"
    ]
)


print("\n" + "=" * 60)
print("SPY NEXT-TRADING-DAY PREDICTION")
print("=" * 60)

print(
    "Latest completed trading day:",
    feature_date.date()
)

print(
    "Latest SPY close:",
    f"${latest_close:,.2f}"
)

print(
    "Predicted next-day direction:",
    prediction_label
)

print(
    "Decision Tree probability of UP:",
    f"{probability_up:.2%}"
)

print(
    "Decision Tree probability of DOWN:",
    f"{probability_down:.2%}"
)

print("=" * 60)


SPY NEXT-TRADING-DAY PREDICTION
Latest completed trading day: 2026-07-10
Latest SPY close: $754.95
Predicted next-day direction: DOWN
Decision Tree probability of UP: 48.33%
Decision Tree probability of DOWN: 51.67%


## Show what rules caused the prediction

In [19]:
# Show the decision path for tomorrow
# =====================================================

node_indicator = final_tree.decision_path(
    X_tomorrow
)

leaf_id = final_tree.apply(
    X_tomorrow
)[0]

sample = X_tomorrow.iloc[0]
feature_names = X_train.columns

print("\nRules followed for this prediction:\n")

for node_id in node_indicator.indices:

    if node_id == leaf_id:
        print(
            f"Reached leaf node {node_id}."
        )
        break

    feature_index = final_tree.tree_.feature[
        node_id
    ]

    threshold = final_tree.tree_.threshold[
        node_id
    ]

    feature_name = feature_names[
        feature_index
    ]

    feature_value = float(
        sample.iloc[feature_index]
    )

    if feature_value <= threshold:
        print(
            f"{feature_name}: "
            f"{feature_value:.6f} "
            f"<= {threshold:.6f}"
        )
    else:
        print(
            f"{feature_name}: "
            f"{feature_value:.6f} "
            f"> {threshold:.6f}"
        )


Rules followed for this prediction:

Feature_130: 0.951680 > 0.936804
Feature_137: 0.018500 <= 0.044513
Feature_179: 0.562112 <= 0.746265
Reached leaf node 8.


In [20]:
# Print complete Decision Tree rules
# =====================================================

tree_rules = export_text(
    final_tree,
    feature_names=list(X_train.columns),
    decimals=6
)

print(tree_rules)

|--- Feature_130 <= 0.936804
|   |--- Feature_066 <= 0.054418
|   |   |--- Feature_180 <= -0.007011
|   |   |   |--- class: 1
|   |   |--- Feature_180 >  -0.007011
|   |   |   |--- class: 1
|   |--- Feature_066 >  0.054418
|   |   |--- class: 0
|--- Feature_130 >  0.936804
|   |--- Feature_137 <= 0.044513
|   |   |--- Feature_179 <= 0.746265
|   |   |   |--- class: 0
|   |   |--- Feature_179 >  0.746265
|   |   |   |--- class: 1
|   |--- Feature_137 >  0.044513
|   |   |--- Feature_227 <= 0.546550
|   |   |   |--- class: 0
|   |   |--- Feature_227 >  0.546550
|   |   |   |--- class: 0



In [21]:
# Show feature importance
# =====================================================

feature_importance = pd.Series(
    final_tree.feature_importances_,
    index=X_train.columns,
    name="Importance"
).sort_values(
    ascending=False
)

used_features = feature_importance[
    feature_importance > 0
]

print("Features used by the tree:")
display(
    used_features.to_frame()
)

Features used by the tree:


,Importance
Feature_179,0.189634
Feature_066,0.188904
Feature_227,0.181896
Feature_137,0.159749
Feature_130,0.143316
Feature_180,0.136502


## Second: Predict if there is a big down tomorrow, using top 5 models

In [22]:
# 1. Automatically select the current top 5 results
# =====================================================

top5_results = (
    results_df
    .head(5)
    .copy()
)

print("Automatically selected top 5 model-target combinations:")

display(
    top5_results[
        [
            "Target",
            "Model",
            "Recall",
            "Precision",
            "F1",
            "Predicted Down Days",
            "Down Hit Rate",
            "Threshold"
        ]
    ]
)

Automatically selected top 5 model-target combinations:


,Target,Model,Recall,Precision,F1,Predicted Down Days,Down Hit Rate,Threshold
0,Top 20% Down,Decision Tree,0.787879,0.181818,0.295455,143,0.468531,-0.006707
1,Top 25% Down,Decision Tree,0.772727,0.242857,0.369565,140,0.485714,-0.004979
2,Top 40% Down,Decision Tree,0.670213,0.411765,0.510121,153,0.470588,-0.001039
3,Top 30% Down,Decision Tree,0.619048,0.288889,0.393939,135,0.459259,-0.003223
4,Top 15% Down,Decision Tree,0.600000,0.148515,0.238095,101,0.465347,-0.008823


In [23]:
# 2. Prepare full historical returns and tomorrow's row
# =====================================================

close = market_close.copy()

# Each date predicts the next trading day's return
future_return = future_return_snapshot.copy()

# Align returns with v_structure
future_return = future_return.reindex(
    v_structure.index
)

# Features only
all_features = v_structure.drop(
    columns=["Target"],
    errors="ignore"
).copy()

all_features = all_features.replace(
    [np.inf, -np.inf],
    np.nan
)

# Latest row whose future outcome is unknown
prediction_dates = future_return[
    future_return.isna()
].index.intersection(
    all_features.dropna().index
)

if len(prediction_dates) == 0:
    raise ValueError(
        "No latest feature row with an unknown future return was found."
    )

prediction_date = prediction_dates.max()

X_tomorrow = all_features.loc[
    [prediction_date]
].copy()


print("Prediction feature date:", prediction_date.date())
print("Prediction shape       :", X_tomorrow.shape)

Prediction feature date: 2026-07-10
Prediction shape       : (1, 253)


In [24]:
# 3. Retrain each selected model on all known history
# =====================================================

live_predictions = []
trained_top5_models = {}

for rank, row in top5_results.reset_index(drop=True).iterrows():

    target_name = row["Target"]
    model_name = row["Model"]

    # Retrieve the target definition automatically
    q = target_settings[target_name]

    # -------------------------------------------------
    # Recompute the target using all known history
    # -------------------------------------------------
    known_returns = future_return.dropna()

    if q == "all_down":
        full_threshold = 0.0

        full_target = (
            future_return < 0
        ).astype(float)

    else:
        # For final live training, calculate the threshold
        # from all currently known historical outcomes
        full_threshold = known_returns.quantile(q)

        full_target = (
            future_return <= full_threshold
        ).astype(float)

    # Preserve unknown future rows as NaN
    full_target.loc[
        future_return.isna()
    ] = np.nan

    # -------------------------------------------------
    # Select complete historical rows
    # -------------------------------------------------
    valid_training_rows = (
        all_features.notna().all(axis=1)
        & full_target.notna()
    )

    X_full_train = all_features.loc[
        valid_training_rows
    ].copy()

    y_full_train = full_target.loc[
        valid_training_rows
    ].astype(int)
    
    # -------------------------------------------------
    # Retrieve the exact model configuration automatically
    # -------------------------------------------------
    if model_name not in models:
        raise KeyError(
            f"{model_name!r} is not available in the models dictionary."
        )

    final_model = clone(
        models[model_name]
    )

    final_model.fit(
        X_full_train,
        y_full_train
    )

    # -------------------------------------------------
    # Predict tomorrow
    # -------------------------------------------------
    live_prediction = int(
        final_model.predict(X_tomorrow)[0]
    )

    probabilities = dict(
        zip(
            final_model.classes_,
            final_model.predict_proba(
                X_tomorrow
            )[0]
        )
    )

    probability_target_down = probabilities.get(
        1,
        0.0
    )

    # Save trained model using rank to avoid name collisions
    model_key = (
        f"Rank_{rank + 1}_"
        f"{target_name}_"
        f"{model_name}"
    )

    trained_top5_models[model_key] = final_model

    live_predictions.append({
        "Rank": rank + 1,
        "Target": target_name,
        "Model": model_name,

        # Previous untouched-test performance
        "Test Recall": row["Recall"],
        "Test Precision": row["Precision"],
        "Test F1": row["F1"],

        # Threshold recalculated using all known history
        "Full-Data Threshold": float(full_threshold),

        "Training Observations": len(X_full_train),
        "Historical Target Rate": y_full_train.mean(),

        "Prediction": (
            "YES"
            if live_prediction == 1
            else "NO"
        ),

        "Probability of Target Down": (
            probability_target_down
        )
    })

## Display the final predictions

In [25]:
# =====================================================
# 4. Final prediction table
# =====================================================

live_predictions_df = pd.DataFrame(
    live_predictions
)

display_df = live_predictions_df.copy()

percentage_columns = [
    "Test Recall",
    "Test Precision",
    "Test F1",
    "Historical Target Rate",
    "Probability of Target Down"
]

for column in percentage_columns:
    display_df[column] = display_df[
        column
    ].map(
        lambda value: f"{value:.1%}"
    )

display_df["Full-Data Threshold"] = (
    display_df["Full-Data Threshold"]
    .map(lambda value: f"{value:.4%}")
)

print(
    f"Predictions using market features through "
    f"{prediction_date.date()}:"
)

display(display_df)

Predictions using market features through 2026-07-10:


,Rank,Target,Model,Test Recall,Test Precision,Test F1,Full-Data Threshold,Training Observations,Historical Target Rate,Prediction,Probability of Target Down
0,1,Top 20% Down,Decision Tree,78.8%,18.2%,29.5%,-0.6473%,1952,20.0%,NO,15.2%
1,2,Top 25% Down,Decision Tree,77.3%,24.3%,37.0%,-0.4551%,1952,25.0%,NO,23.4%
2,3,Top 40% Down,Decision Tree,67.0%,41.2%,51.0%,-0.0916%,1952,40.0%,NO,44.9%
3,4,Top 30% Down,Decision Tree,61.9%,28.9%,39.4%,-0.3064%,1952,30.0%,NO,33.8%
4,5,Top 15% Down,Decision Tree,60.0%,14.9%,23.8%,-0.8499%,1952,15.0%,NO,14.5%


# Prediction Tracking

In [26]:
# ============================================================
# 1. Settings
# ============================================================

PREDICTION_LOG_FILE = "spy_prediction_history.csv"


# ============================================================
# 2. Prepare SPY prices and actual next-day returns
# ============================================================

spy_clean = pd.DataFrame({"Close": market_close}).copy()
spy_clean = spy_clean.sort_index()

close = spy_clean["Close"].squeeze()

# For a row dated t:
# actual_next_return[t] = Close[t+1] / Close[t] - 1
actual_next_return = close.shift(-1) / close - 1


# ============================================================
# 3. Identify the feature date used for today's prediction
# ============================================================

if X_tomorrow.empty:
    raise ValueError(
        "X_tomorrow is empty. Build the latest prediction row first."
    )

prediction_feature_date = pd.Timestamp(
    X_tomorrow.index[-1]
)

if prediction_feature_date not in close.index:
    raise ValueError(
        f"Prediction feature date {prediction_feature_date.date()} "
        "was not found in SPY price data."
    )

prediction_close = float(
    close.loc[prediction_feature_date]
)

print("=" * 70)
print("SPY LIVE PREDICTION TRACKER")
print("=" * 70)

print(
    "Latest feature date:",
    prediction_feature_date.date()
)

print(
    "SPY close:",
    f"${prediction_close:,.2f}"
)


# ============================================================
# 4. Create today's Up/Down prediction
# ============================================================

up_down_prediction = int(
    final_tree.predict(X_tomorrow)[0]
)

up_down_probabilities = dict(
    zip(
        final_tree.classes_,
        final_tree.predict_proba(X_tomorrow)[0]
    )
)

probability_down = float(
    up_down_probabilities.get(0, 0.0)
)

probability_up = float(
    up_down_probabilities.get(1, 0.0)
)

up_down_label = (
    "UP"
    if up_down_prediction == 1
    else "DOWN"
)

# Store probability of the direction actually predicted
up_down_prediction_probability = (
    probability_up
    if up_down_label == "UP"
    else probability_down
)

up_down_row = {
    "Prediction Date": prediction_feature_date,
    "Model Rank": 0,
    "Target": "Up/Down",
    "Model": "Decision Tree",
    "Threshold": 0.0,
    "Prediction": up_down_label,
    "Probability of Prediction": up_down_prediction_probability,

    # Keep both directional probabilities
    "Probability Up": probability_up,
    "Probability Down": probability_down,

    # These may be added later if you save the
    # Up/Down model's test statistics
    "Test Recall": np.nan,
    "Test Precision": np.nan,
    "Test F1": np.nan,

    # Filled after the next trading-day close becomes available
    "Actual Next-Day Return": np.nan,
    "Actual Result": np.nan,
    "Correct": np.nan
}


# ============================================================
# 5. Create today's top-five event prediction rows
# ============================================================

required_live_columns = [
    "Rank",
    "Target",
    "Model",
    "Full-Data Threshold",
    "Prediction",
    "Probability of Target Down",
    "Test Recall",
    "Test Precision",
    "Test F1"
]

missing_columns = [
    col
    for col in required_live_columns
    if col not in live_predictions_df.columns
]

if missing_columns:
    raise KeyError(
        "live_predictions_df is missing these columns: "
        f"{missing_columns}"
    )

top5_prediction_rows = []

for _, row in live_predictions_df.iterrows():

    top5_prediction_rows.append({
        "Prediction Date": prediction_feature_date,
        "Model Rank": int(row["Rank"]),
        "Target": str(row["Target"]),
        "Model": str(row["Model"]),
        "Threshold": float(row["Full-Data Threshold"]),

        # YES means the Top-X% down event is predicted
        # NO means the event is not predicted
        "Prediction": str(row["Prediction"]),

        "Probability of Prediction": float(
            row["Probability of Target Down"]
        ),

        # Not applicable to the event models
        "Probability Up": np.nan,
        "Probability Down": np.nan,

        "Test Recall": float(row["Test Recall"]),
        "Test Precision": float(row["Test Precision"]),
        "Test F1": float(row["Test F1"]),

        "Actual Next-Day Return": np.nan,
        "Actual Result": np.nan,
        "Correct": np.nan
    })


# ============================================================
# 6. Combine today's six predictions
# ============================================================

today_predictions = pd.DataFrame(
    [up_down_row] + top5_prediction_rows
)

today_predictions["Prediction Date"] = pd.to_datetime(
    today_predictions["Prediction Date"]
)

today_predictions = today_predictions.sort_values(
    "Model Rank"
).reset_index(drop=True)


# ============================================================
# 7. Load previous prediction history
# ============================================================

if os.path.exists(PREDICTION_LOG_FILE):

    prediction_history = pd.read_csv(
        PREDICTION_LOG_FILE,
        parse_dates=["Prediction Date"]
    )

    print(
        f"\nLoaded {len(prediction_history)} saved prediction rows."
    )

else:

    prediction_history = pd.DataFrame(
        columns=today_predictions.columns
    )

    prediction_history["Prediction Date"] = pd.to_datetime(
        prediction_history["Prediction Date"]
    )

    print(
        "\nNo existing history file was found. "
        "A new one will be created automatically."
    )


# ============================================================
# 8. Make old CSV versions compatible
# ============================================================

for column in today_predictions.columns:

    if column not in prediction_history.columns:
        prediction_history[column] = np.nan

# Keep a consistent column order
prediction_history = prediction_history[
    today_predictions.columns
]

prediction_history["Prediction Date"] = pd.to_datetime(
    prediction_history["Prediction Date"]
)


# ============================================================
# 9. Evaluate all pending previous predictions
# ============================================================

number_updated = 0

for idx, row in prediction_history.iterrows():

    prediction_date = pd.Timestamp(
        row["Prediction Date"]
    )

    # Do not evaluate an already completed prediction
    if pd.notna(row["Actual Next-Day Return"]):
        continue

    # The original prediction date must exist in SPY data
    if prediction_date not in actual_next_return.index:
        continue

    realized_return = actual_next_return.loc[
        prediction_date
    ]

    # NaN means the next trading-day close is not available yet
    if pd.isna(realized_return):
        continue

    realized_return = float(realized_return)

    target_name = str(row["Target"])
    prediction = str(row["Prediction"])

    # --------------------------------------------------------
    # Up/Down prediction
    # --------------------------------------------------------

    if target_name == "Up/Down":

        actual_result = (
            "UP"
            if realized_return > 0
            else "DOWN"
        )

    # --------------------------------------------------------
    # Top-X% down-event prediction
    # --------------------------------------------------------

    else:

        threshold = float(row["Threshold"])

        event_occurred = (
            realized_return <= threshold
        )

        actual_result = (
            "YES"
            if event_occurred
            else "NO"
        )

    correct = (
        "CORRECT"
        if prediction == actual_result
        else "WRONG"
    )

    prediction_history.loc[
        idx,
        "Actual Next-Day Return"
    ] = realized_return

    prediction_history.loc[
        idx,
        "Actual Result"
    ] = actual_result

    prediction_history.loc[
        idx,
        "Correct"
    ] = correct

    number_updated += 1


print(
    f"Previous prediction rows updated with actual results: "
    f"{number_updated}"
)


# ============================================================
# 10. Prevent duplicate predictions for the same date
# ============================================================

# A unique prediction is identified by:
# Prediction Date + Target + Model
#
# We do not rely on Model Rank because ranks can change.

if not prediction_history.empty:

    current_prediction_keys = set(
        zip(
            today_predictions["Prediction Date"],
            today_predictions["Target"],
            today_predictions["Model"]
        )
    )

    keep_existing_rows = []

    for _, old_row in prediction_history.iterrows():

        old_key = (
            pd.Timestamp(old_row["Prediction Date"]),
            old_row["Target"],
            old_row["Model"]
        )

        keep_existing_rows.append(
            old_key not in current_prediction_keys
        )

    prediction_history = prediction_history.loc[
        keep_existing_rows
    ].copy()


# ============================================================
# 11. Add today's six predictions
# ============================================================

prediction_history = pd.concat(
    [
        prediction_history,
        today_predictions
    ],
    ignore_index=True
)

prediction_history["Prediction Date"] = pd.to_datetime(
    prediction_history["Prediction Date"]
)

prediction_history = prediction_history.sort_values(
    by=[
        "Prediction Date",
        "Model Rank"
    ],
    ascending=[
        False,
        True
    ]
).reset_index(drop=True)


# ============================================================
# 12. Save the complete prediction history
# ============================================================

prediction_history.to_csv(
    PREDICTION_LOG_FILE,
    index=False
)

print(
    f"Prediction history saved to: {PREDICTION_LOG_FILE}"
)

print(
    "Current working folder:",
    os.getcwd()
)


# ============================================================
# 13. Formatting helper
# ============================================================

def format_prediction_table(df):
    """
    Return a readable display copy without changing
    the numeric values stored in the CSV.
    """

    formatted = df.copy()

    formatted["Prediction Date"] = pd.to_datetime(
        formatted["Prediction Date"]
    ).dt.date

    formatted["Threshold"] = formatted[
        "Threshold"
    ].apply(
        lambda value:
        f"{value:.2%}"
        if pd.notna(value)
        else ""
    )

    formatted["Probability of Prediction"] = formatted[
        "Probability of Prediction"
    ].apply(
        lambda value:
        f"{value:.1%}"
        if pd.notna(value)
        else ""
    )

    formatted["Probability Up"] = formatted[
        "Probability Up"
    ].apply(
        lambda value:
        f"{value:.1%}"
        if pd.notna(value)
        else ""
    )

    formatted["Probability Down"] = formatted[
        "Probability Down"
    ].apply(
        lambda value:
        f"{value:.1%}"
        if pd.notna(value)
        else ""
    )

    formatted["Actual Next-Day Return"] = formatted[
        "Actual Next-Day Return"
    ].apply(
        lambda value:
        f"{value:.2%}"
        if pd.notna(value)
        else "Pending"
    )

    for column in [
        "Test Recall",
        "Test Precision",
        "Test F1"
    ]:

        formatted[column] = formatted[
            column
        ].apply(
            lambda value:
            f"{value:.1%}"
            if pd.notna(value)
            else ""
        )

    formatted["Actual Result"] = formatted[
        "Actual Result"
    ].fillna("Pending")

    formatted["Correct"] = formatted[
        "Correct"
    ].fillna("Pending")

    return formatted


# ============================================================
# 14. Show today's six predictions
# ============================================================

today_summary_display = format_prediction_table(
    today_predictions
)

print(
    "\nToday's six-model prediction summary:"
)

display(
    today_summary_display[
        [
            "Prediction Date",
            "Model Rank",
            "Target",
            "Model",
            "Threshold",
            "Prediction",
            "Probability of Prediction",
            "Probability Up",
            "Probability Down",
            "Test Recall",
            "Test Precision",
            "Test F1"
        ]
    ]
)


# ============================================================
# 15. Show the most recent completed prediction date
# ============================================================

completed_history = prediction_history[
    prediction_history["Actual Next-Day Return"].notna()
].copy()

if completed_history.empty:

    print(
        "\nNo previous live predictions have been evaluated yet."
    )

else:

    latest_completed_date = completed_history[
        "Prediction Date"
    ].max()

    latest_completed = completed_history[
        completed_history["Prediction Date"]
        == latest_completed_date
    ].copy()

    latest_completed_display = format_prediction_table(
        latest_completed
    )

    print(
        "\nMost recently evaluated prediction summary:",
        pd.Timestamp(latest_completed_date).date()
    )

    display(
        latest_completed_display[
            [
                "Prediction Date",
                "Model Rank",
                "Target",
                "Model",
                "Threshold",
                "Prediction",
                "Probability of Prediction",
                "Actual Next-Day Return",
                "Actual Result",
                "Correct"
            ]
        ]
    )


# ============================================================
# 16. Show the full prediction history
# ============================================================

full_history_display = format_prediction_table(
    prediction_history
)

print("\nComplete SPY live-prediction history:")

display(
    full_history_display[
        [
            "Prediction Date",
            "Model Rank",
            "Target",
            "Model",
            "Threshold",
            "Prediction",
            "Probability of Prediction",
            "Actual Next-Day Return",
            "Actual Result",
            "Correct",
            "Test Recall",
            "Test Precision",
            "Test F1"
        ]
    ]
)


# ============================================================
# 17. Live accuracy by target and model
# ============================================================

completed_predictions = prediction_history[
    prediction_history["Correct"].isin(
        ["CORRECT", "WRONG"]
    )
].copy()

if completed_predictions.empty:

    print(
        "\nNo completed live predictions are available "
        "for a performance summary."
    )

else:

    completed_predictions["Correct Number"] = (
        completed_predictions["Correct"]
        == "CORRECT"
    ).astype(int)

    live_performance = (
        completed_predictions
        .groupby(
            [
                "Target",
                "Model"
            ],
            as_index=False
        )
        .agg(
            Predictions_Evaluated=(
                "Correct Number",
                "size"
            ),
            Correct_Predictions=(
                "Correct Number",
                "sum"
            ),
            Live_Accuracy=(
                "Correct Number",
                "mean"
            ),
            Average_Prediction_Probability=(
                "Probability of Prediction",
                "mean"
            )
        )
        .sort_values(
            by=[
                "Live_Accuracy",
                "Predictions_Evaluated"
            ],
            ascending=[
                False,
                False
            ]
        )
    )

    live_performance_display = live_performance.copy()

    live_performance_display[
        "Live_Accuracy"
    ] = live_performance_display[
        "Live_Accuracy"
    ].map(
        lambda value: f"{value:.1%}"
    )

    live_performance_display[
        "Average_Prediction_Probability"
    ] = live_performance_display[
        "Average_Prediction_Probability"
    ].map(
        lambda value: f"{value:.1%}"
    )

    print("\nLive performance by target and model:")

    display(live_performance_display)


# ============================================================
# 18. Coverage summary
# ============================================================

unique_prediction_dates = (
    prediction_history["Prediction Date"]
    .dropna()
    .sort_values()
    .unique()
)

print("\nPrediction-log coverage:")

if len(unique_prediction_dates) == 0:

    print("No prediction dates are stored.")

else:

    print(
        "First prediction date:",
        pd.Timestamp(unique_prediction_dates[0]).date()
    )

    print(
        "Latest prediction date:",
        pd.Timestamp(unique_prediction_dates[-1]).date()
    )

    print(
        "Number of dates on which predictions were actually made:",
        len(unique_prediction_dates)
    )

    print(
        "\nMissing trading days are not automatically backfilled. "
        "They remain absent because no live prediction was made "
        "on those dates."
    )

SPY LIVE PREDICTION TRACKER
Latest feature date: 2026-07-10
SPY close: $754.95

Loaded 12 saved prediction rows.
Previous prediction rows updated with actual results: 0
Prediction history saved to: spy_prediction_history.csv
Current working folder: /home/jovyan/Projects/Stock-Forecasting/Demo

Today's six-model prediction summary:


,Prediction Date,Model Rank,Target,Model,Threshold,Prediction,Probability of Prediction,Probability Up,Probability Down,Test Recall,Test Precision,Test F1
0,2026-07-10,0,Up/Down,Decision Tree,0.00%,DOWN,51.7%,48.3%,51.7%,,,
1,2026-07-10,1,Top 20% Down,Decision Tree,-0.65%,NO,15.2%,,,78.8%,18.2%,29.5%
2,2026-07-10,2,Top 25% Down,Decision Tree,-0.46%,NO,23.4%,,,77.3%,24.3%,37.0%
3,2026-07-10,3,Top 40% Down,Decision Tree,-0.09%,NO,44.9%,,,67.0%,41.2%,51.0%
4,2026-07-10,4,Top 30% Down,Decision Tree,-0.31%,NO,33.8%,,,61.9%,28.9%,39.4%
5,2026-07-10,5,Top 15% Down,Decision Tree,-0.85%,NO,14.5%,,,60.0%,14.9%,23.8%



Most recently evaluated prediction summary: 2026-07-09


,Prediction Date,Model Rank,Target,Model,Threshold,Prediction,Probability of Prediction,Actual Next-Day Return,Actual Result,Correct
6,2026-07-09,0,Up/Down,Decision Tree,0.00%,DOWN,51.6%,0.43%,UP,WRONG
7,2026-07-09,1,Top 25% Down,Decision Tree,-0.44%,NO,21.3%,0.43%,NO,CORRECT
8,2026-07-09,2,Top 20% Down,Decision Tree,-0.64%,NO,15.5%,0.43%,NO,CORRECT
9,2026-07-09,3,Top 40% Down,Decision Tree,-0.09%,YES,54.3%,0.43%,NO,WRONG
10,2026-07-09,4,Top 30% Down,Decision Tree,-0.30%,NO,35.7%,0.43%,NO,CORRECT
11,2026-07-09,5,Top 15% Down,Decision Tree,-0.84%,NO,7.6%,0.43%,NO,CORRECT



Complete SPY live-prediction history:


,Prediction Date,Model Rank,Target,Model,Threshold,Prediction,Probability of Prediction,Actual Next-Day Return,Actual Result,Correct,Test Recall,Test Precision,Test F1
0,2026-07-10,0,Up/Down,Decision Tree,0.00%,DOWN,51.7%,Pending,Pending,Pending,,,
1,2026-07-10,1,Top 20% Down,Decision Tree,-0.65%,NO,15.2%,Pending,Pending,Pending,78.8%,18.2%,29.5%
2,2026-07-10,2,Top 25% Down,Decision Tree,-0.46%,NO,23.4%,Pending,Pending,Pending,77.3%,24.3%,37.0%
3,2026-07-10,3,Top 40% Down,Decision Tree,-0.09%,NO,44.9%,Pending,Pending,Pending,67.0%,41.2%,51.0%
4,2026-07-10,4,Top 30% Down,Decision Tree,-0.31%,NO,33.8%,Pending,Pending,Pending,61.9%,28.9%,39.4%
5,2026-07-10,5,Top 15% Down,Decision Tree,-0.85%,NO,14.5%,Pending,Pending,Pending,60.0%,14.9%,23.8%
6,2026-07-09,0,Up/Down,Decision Tree,0.00%,DOWN,51.6%,0.43%,UP,WRONG,,,
7,2026-07-09,1,Top 25% Down,Decision Tree,-0.44%,NO,21.3%,0.43%,NO,CORRECT,74.5%,25.0%,37.4%
8,2026-07-09,2,Top 20% Down,Decision Tree,-0.64%,NO,15.5%,0.43%,NO,CORRECT,69.7%,16.0%,26.0%
9,2026-07-09,3,Top 40% Down,Decision Tree,-0.09%,YES,54.3%,0.43%,NO,WRONG,61.7%,44.3%,51.6%



Live performance by target and model:


,Target,Model,Predictions_Evaluated,Correct_Predictions,Live_Accuracy,Average_Prediction_Probability
0,Top 15% Down,Decision Tree,1,1,100.0%,7.6%
1,Top 20% Down,Decision Tree,1,1,100.0%,15.5%
2,Top 25% Down,Decision Tree,1,1,100.0%,21.3%
3,Top 30% Down,Decision Tree,1,1,100.0%,35.7%
4,Top 40% Down,Decision Tree,1,0,0.0%,54.3%
5,Up/Down,Decision Tree,1,0,0.0%,51.6%



Prediction-log coverage:
First prediction date: 2026-07-09
Latest prediction date: 2026-07-10
Number of dates on which predictions were actually made: 2

Missing trading days are not automatically backfilled. They remain absent because no live prediction was made on those dates.


# Final Conclusion

## Summary and Results of This Demo

This project began as an attempt to forecast the exact next-day closing price of SPY. After AutoReg, ARIMA, and SARIMAX failed to meaningfully outperform a simple persistence forecast, I reformulated the problem as a classification system for predicting next-day market direction and identifying unusually large downside moves.

The experiments showed that predicting the direction of every trading day is extremely difficult. Most models struggled to meaningfully outperform the approximately 55% Always Up baseline. Regularized linear models and Random Forest often predicted Up almost every day, while more complex models such as XGBoost tended to overfit. A small Decision Tree was more competitive, stable, and interpretable, demonstrating that additional complexity cannot compensate for weak or noisy predictive information.

The most promising results came from predicting unusually large down days. The compressed candlestick-structure features performed better than the much larger extended feature set, suggesting that carefully designed features can be more useful than simply adding more information. Although these models still produce false warnings, they may provide valuable information for managing market exposure.

In practice, I could use this system to hold SPY during normal market conditions and reduce exposure when the predicted probability of a large decline becomes unusually high. When downside risk is low and the models indicate strong bullish conditions, I could also consider a controlled allocation to SPXL, a leveraged ETF designed to produce approximately three times the daily return of the S&P 500. Because leverage magnifies both gains and losses, any SPXL strategy would require strict position limits, risk controls, and separate evaluation.

## Development of a Full Machine-Learning Trading System

This SPY forecasting model is only one part of a broader machine-learning trading system that I am continuously developing. I am also building similar models for individual stocks that I follow and trade. My longer-term goal is to create specialized models for different quantitative-trading objectives, including:

- Direction prediction: Estimate the probability that a stock or market will rise or fall over the next day, week, or another specified horizon.
- Return-magnitude prediction: Estimate the expected size of a future return so that trades are considered only when the potential move is sufficiently large.
- Large-move probability: Estimate the probability of an unusually large decline, rally, or other tail event to support hedging and exposure management.
- Volatility forecasting: Predict future volatility to guide position sizing, option selection, leverage, and risk limits.
- Market-regime detection: Identify bullish, bearish, sideways, low-volatility, and high-volatility conditions so that the trading strategy can adapt to the current environment.
- Cross-sectional ranking: Rank stocks by their expected relative performance to identify stronger opportunities and avoid weaker ones.
- Mean-reversion detection: Determine when an unusually large price movement may reverse, creating an opportunity to buy temporary weakness or sell temporary strength.
- Trend-persistence prediction: Estimate whether an existing trend is likely to continue, supporting momentum and trend-following strategies.
- Relative-value analysis: Detect unusual divergences between related stocks or assets for pairs trading and statistical-arbitrage strategies.
- Portfolio allocation: Determine how capital should be distributed among stocks, bonds, gold, cash, leveraged ETFs, and other assets to improve risk-adjusted performance.
- Risk modeling: Estimate drawdown probability, Value at Risk, correlation, beta, and portfolio-level exposure to limit losses and avoid excessive concentration.

## Closing Words

My goal is not to create one model that answers every trading question, but to build a connected system of specialized models that helps determine what to trade, when to trade, how much to invest, and when to reduce risk.

This public notebook is a demo version that demonstrates my approach to feature design, data-leakage prevention, model comparison, prediction interpretation, and applying machine-learning results to practical decisions. It also serves as a foundation for a broader system that I plan to continue improving through walk-forward testing, live prediction tracking, and disciplined risk management.